# Validation Analysis - Smart Real Estate DSS

Notebook này dùng để trình bày phần **data validation** cho hệ thống tư vấn bất động sản.

Mục tiêu:
- Load tập `validation_50_scenarios.json`.
- Load kết quả `validation_summary.json`.
- Trình bày các metric Top-K recommendation.
- Phân tích theo nhóm người dùng và edge cases.
- Ghi rõ giới hạn: đây là **synthetic validation**, chưa phải human-labeled validation.

## 1. Load Data

Notebook đọc lại các file đã được sinh bởi pipeline validation:

- `data/go_vap_enriched.json`: tập BĐS đã enrich POI.
- `data/validation_50_scenarios.json`: 50 synthetic user preference scenarios.
- `outputs/validation_summary.json`: metric tổng hợp sau khi chạy `src/eval/evaluate_pipeline.py`.

In [ ]:
from pathlib import Path
import json

import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

DATA_DIR = ROOT / "data"
OUTPUT_DIR = ROOT / "outputs"

with open(DATA_DIR / "go_vap_enriched.json", "r", encoding="utf-8") as f:
    properties = json.load(f)

with open(DATA_DIR / "validation_50_scenarios.json", "r", encoding="utf-8") as f:
    scenarios = json.load(f)

with open(OUTPUT_DIR / "validation_summary.json", "r", encoding="utf-8") as f:
    summary = json.load(f)

print(f"Properties: {len(properties)}")
print(f"Validation scenarios: {len(scenarios)}")
print(f"Archetypes: {len(summary['per_archetype'])}")

## 2. Global Metrics

Các metric này đánh giá recommendation dạng Top-K, phù hợp hơn accuracy truyền thống vì bài toán DSS không có nhãn đúng/sai tuyệt đối.

In [ ]:
global_metrics = summary["global_metrics"]

global_df = pd.DataFrame([
    {"metric": "Total scenarios", "value": global_metrics["total_scenarios"]},
    {"metric": "Constraint Satisfaction Rate", "value": global_metrics["avg_csr"]},
    {"metric": "Precision@3", "value": global_metrics["avg_precision_3"]},
    {"metric": "Precision@5", "value": global_metrics["avg_precision_5"]},
    {"metric": "Recall@3", "value": global_metrics["avg_recall_3"]},
    {"metric": "Recall@5", "value": global_metrics["avg_recall_5"]},
    {"metric": "NDCG@3", "value": global_metrics["avg_ndcg_3"]},
    {"metric": "NDCG@5", "value": global_metrics["avg_ndcg_5"]},
    {"metric": "MAP", "value": global_metrics["mean_avg_precision"]},
])

display(global_df)

## 3. Per-Archetype Breakdown

Phân tích theo archetype giúp phát hiện nhóm người dùng nào có kết quả thấp hơn để điều chỉnh dữ liệu, constraint hoặc trọng số scoring.

In [ ]:
arch_df = (
    pd.DataFrame(summary["per_archetype"])
    .T
    .reset_index()
    .rename(columns={"index": "archetype"})
)

metric_cols = ["csr", "p3", "p5", "r3", "r5", "n3", "n5", "ap"]
display(arch_df[["archetype", *metric_cols]])

In [ ]:
plot_df = arch_df.set_index("archetype")[["p5", "r5", "n5", "ap"]]

ax = plot_df.plot(kind="bar", figsize=(10, 5), rot=25)
ax.set_ylim(0, 1.05)
ax.set_title("Validation metrics by user archetype")
ax.set_ylabel("Score")
ax.set_xlabel("Archetype")
ax.legend(title="Metric")
plt.tight_layout()
plt.show()

## 4. Edge Cases

Edge cases quan trọng với DSS vì hệ thống cần xử lý được cả trường hợp không có BĐS phù hợp hoặc số lượng candidate ít hơn Top 5.

In [ ]:
edge_cases = summary["edge_cases"]

zero_df = pd.DataFrame({"scenario_id": edge_cases["zero_candidates"]})
few_df = pd.DataFrame(edge_cases["few_candidates"])

print("Zero-candidate scenarios:")
display(zero_df)

print("Few-candidate scenarios:")
display(few_df)

## 5. Scenario Dataset Overview

Bảng dưới đây giúp kiểm tra nhanh phân bố scenario, budget, số phòng ngủ tối thiểu và ground-truth Top 5.

In [ ]:
scenario_rows = []
for sc in scenarios:
    scenario_rows.append({
        "scenario_id": sc["scenario_id"],
        "archetype": sc["archetype"],
        "name": sc["name"],
        "budget_max_million": sc["hard_constraints"]["budget_max_million"],
        "min_bedrooms": sc["hard_constraints"]["min_bedrooms"],
        "candidates_after_filter": sc.get("candidates_after_filter"),
        "ground_truth_top5": ", ".join(sc["ground_truth_top5"]),
    })

scenario_df = pd.DataFrame(scenario_rows)
display(scenario_df.head(10))

display(
    scenario_df.groupby("archetype")
    .agg(
        scenarios=("scenario_id", "count"),
        avg_budget_million=("budget_max_million", "mean"),
        avg_candidates=("candidates_after_filter", "mean"),
    )
    .round(2)
)

## 6. Interpretation For DSS Report

**Kết luận hiện tại:**

- Hệ thống đạt CSR cao, nghĩa là hard constraints như ngân sách và số phòng ngủ được đảm bảo.
- Các metric Top-K cho thấy pipeline ranking hoạt động ổn định trên synthetic scenarios.
- Một số edge cases không có đủ candidates, phản ánh giới hạn của tập 37 BĐS Gò Vấp.

**Giới hạn quan trọng:**

`validation_50_scenarios.json` là synthetic validation set. Ground truth Top 5 được sinh từ reference scorer, nên chưa chứng minh recommendation đúng với người mua thật. Bộ này nên được trình bày là technical validation / stress test / regression test.

## 7. Next Step: Human-Labeled Validation

Để tăng tính đúng đắn cho môn DSS with Data, nhóm nên tạo thêm tập validation có nhãn từ người dùng hoặc người chấm.

Schema đề xuất:

```json
{
  "scenario_id": "REAL_001",
  "persona": "family",
  "user_need": "Gia đình có 2 con nhỏ, ngân sách dưới 6 tỷ, cần 3 phòng ngủ, ưu tiên gần trường và công viên.",
  "budget_max_million": 6000,
  "min_bedrooms": 3,
  "importance": {
    "school": 5,
    "park": 4,
    "hospital": 3,
    "supermarket": 4,
    "transport": 2
  },
  "human_relevance": {
    "GV_008": 5,
    "GV_035": 4,
    "GV_029": 4,
    "GV_031": 3,
    "GV_007": 2
  }
}
```

Human relevance score có thể dùng để tính NDCG@5, MAP và average relevance độc lập hơn so với synthetic ground truth.